In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init
import numpy as np
import random
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import gc
import os
import math
from torchvision.datasets import EMNIST
import string
import torchvision
from torchvision import datasets, transforms
from tqdm import tqdm
from torchvision.utils import make_grid
from torch.utils.data import DataLoader

In [ ]:
english_letters = " " + string.ascii_lowercase

# Dataset

In [ ]:
BATCH_SIZE = 1024

In [ ]:
class Transpose(object):
    def __call__(self, pic):
        return pic.transpose(-1, -2)

transform = transforms.Compose([
    transforms.Grayscale(),        
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    Transpose(),
])

trainset = EMNIST(root='../data', split='letters', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, drop_last=True)

# Display sample of the dataset

In [ ]:

def show_tensor_grid(x_0, nrow=8, title=''):
    grid = make_grid(x_0.detach().cpu(), nrow=nrow, normalize=True)
    np_grid = grid.permute(1, 2, 0).numpy()  
    
    plt.title(title)
    plt.imshow(np_grid.squeeze(), cmap='gray')
    plt.axis('off')
    plt.show()

In [ ]:
data_iter = iter(trainloader)
images, labels = next(data_iter)

show_tensor_grid(images[:16], 4, title="Sample")

# Constants

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
def get_beta_schedule(schedule_type="linear", timesteps=1000, beta_start=1e-5, beta_end=0.02):
    if schedule_type == "linear":
        return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)

    elif schedule_type == "cosine":
        def alpha_bar(t, T):
            return np.cos((t / T + 0.008) / 1.008 * np.pi / 2) ** 2

        betas = []
        for i in range(timesteps):
            t1 = i / timesteps
            t2 = (i + 1) / timesteps
            beta = min(1 - alpha_bar(t2, 1) / alpha_bar(t1, 1), 0.999)
            betas.append(beta)

        return torch.tensor(betas, dtype=torch.float32)

    else:
        raise ValueError("Unknown schedule type. Use 'linear' or 'cosine'.")

In [ ]:
T = 2000

B = get_beta_schedule("cosine", timesteps=T).to(device)  
A = 1.0 - B                                              
A_SQRT = A.sqrt()                                        
A_PROD = torch.cumprod(A, dim=0)                         
A_PROD_SQRT = A_PROD.sqrt().clamp(min=1e-8)              

# Model

In [ ]:
class DoubleCNN(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(0.1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )
        self.time_mlp = nn.Linear(time_emb_dim, out_channels)

    def forward(self, x, t_emb):
        h = self.net(x)
        t = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        return h + t

class UNET(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim=128, features=[64, 128, 256, 512, 1024]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        
        for feature in features:
            self.downs.append(DoubleCNN(in_channels, feature, time_emb_dim))
            in_channels = feature

        
        self.bottleneck = DoubleCNN(features[-1], features[-1]*2, time_emb_dim)

        
        rev_features = features[::-1]
        for i in range(len(rev_features)):
            self.ups.append(nn.ConvTranspose2d(
                rev_features[i]*2 if i==0 else rev_features[i-1],
                rev_features[i],
                kernel_size=2,
                stride=2
            ))
            self.ups.append(DoubleCNN(rev_features[i]*2, rev_features[i], time_emb_dim))

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x, t_emb):
        skip_connections = []

        for down in self.downs:
            x = down(x, t_emb)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x, t_emb)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skip_connections[i // 2]
            if x.shape != skip.shape:
                x = TF.resize(x, size=skip.shape[2:])
            x = torch.cat((skip, x), dim=1)
            x = self.ups[i+1](x, t_emb)

        return self.final(x)

class TimestepEmbedding(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
        )
        self.emb_dim = emb_dim

    def forward(self, t):
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return self.mlp(emb)

class UNET_DIFFUSION(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim=128, features=[64, 128, 256, 512, 1024], num_classes=10):
        super().__init__()
        self.time_encoder = TimestepEmbedding(time_emb_dim)
        self.class_encoder = nn.Embedding(num_classes, time_emb_dim)
        self.unet = UNET(in_channels, out_channels, time_emb_dim, features)

    def forward(self, x, t, labels=None):
        t_emb = self.time_encoder(t)
        if labels is not None:
            label_emb = self.class_encoder(labels)
        else:
            label_emb = torch.zeros_like(t_emb)
        return self.unet(x, t_emb + label_emb)

In [ ]:
model = UNET_DIFFUSION(1,1, time_emb_dim=64, features=[32, 64, 128], num_classes=30).to(device)

if 'model_english.pt' in os.listdir('.'):
    model.load_state_dict(torch.load('model_english.pt'))

# Training Loop

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-5)

In [ ]:
model.to(device)
model.train()
loss_fn = nn.MSELoss()
num_epochs = 30

for epoch in range(num_epochs):
    for i, (real_images, labels) in enumerate(trainloader):
        real_images = real_images.to(device)  
        labels = labels.to(device)

        # Train model to generate 'space' char
        if random.random() < 0.05:
            labels *= 0
            real_images *= 0

        t = torch.randint(low=1, high=T - 1, size=(real_images.size(0),), device=device)

        noise = torch.randn_like(real_images)

        a_sqrt = A_PROD_SQRT[t].view(-1, 1, 1, 1)
        one_minus_a = (1 - A_PROD[t]).sqrt().view(-1, 1, 1, 1)

        x_t = a_sqrt * real_images + one_minus_a * noise

        noise_pred = model(x_t.float(), t.float(), labels)

        mse_loss = loss_fn(noise_pred, noise)

        cosine_loss = 1 - F.cosine_similarity(
            noise_pred.view(noise_pred.size(0), -1),
            noise.view(noise.size(0), -1),
            dim=1
        ).mean()

        loss = mse_loss + cosine_loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if i % 30 == 0 or i == 10:
            print(f"Epoch {epoch+1}/{num_epochs} Step {i}/{len(trainloader)} Loss: {loss.item():.4f}", end="\r")
            gc.collect()
            torch.cuda.empty_cache()

model.eval()

Save the model after training

In [224]:
torch.save(model.state_dict(), f'model_english.pt')

# Sampling

In [253]:
@torch.no_grad()
def sample_ddpm(model: nn.Module, labels, device=torch.device('cuda')):
    model.eval()
    n = labels.size(0)
    x_t = torch.randn((n, 1, 32, 32), device=device)

    small_std = 0.002

    for t in reversed(range(T)):
        t_tensor = torch.full((n,), t, dtype=torch.float32, device=device)

        noise_pred = model(x_t, t_tensor, labels)

        a_prod_sqrt = A_PROD_SQRT[t].reshape(1, 1, 1, 1)
        one_minus_a_prod_sqrt = (1 - A_PROD[t]).sqrt().reshape(1, 1, 1, 1)

        x_0_pred = (x_t - one_minus_a_prod_sqrt * noise_pred) / a_prod_sqrt

        if t > 0:
            coef1 = ((A_PROD_SQRT[t - 1] * B[t]) / (1 - A_PROD[t])).reshape(1, 1, 1, 1)
            coef2 = ((A_SQRT[t] * (1 - A_PROD[t - 1])) / (1 - A_PROD[t])).reshape(1, 1, 1, 1)
            noise = torch.randn_like(x_t) * B[t].sqrt().reshape(1, 1, 1, 1)
            x_t = coef1 * x_0_pred + coef2 * x_t + noise

            if t > 50:
                small_noise = torch.randn_like(x_t) * small_std
                x_t += small_noise
        else:
            x_t = x_0_pred

    return x_t

In [254]:
lbls = torch.randint(0, len(english_letters) - 1, (16,)).to(device)
x_0 = sample_ddpm(model, labels=lbls)

In [ ]:
img = x_0[0, 0].detach().cpu().numpy()  

plt.title(english_letters[lbls[0]])
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
print('[!] Letters:')
print([english_letters[i.item()] for i in lbls[:len(x_0)]])


show_tensor_grid((((x_0 > 0.3)*x_0)), 4)

Drawing image of a word

In [ ]:
word = 'home made'
word_labels = [english_letters.lower().index(c.lower()) for c in word]
word_labels = torch.tensor(word_labels).to(device)
model.eval()
x_word = sample_ddpm(model, labels=word_labels)

show_tensor_grid((((x_word > 0.3)*x_word)), len(x_word))